# Customer Churn Prediction using built ANN

In [73]:
from tensorflow.keras.models import load_model
import pickle

import numpy as np
import pandas as pd

In [74]:
import os

In [75]:
gender_transformation_path = os.path.join('feature_transformation_artifacts', 'gender_ohe.pkl')
geography_transformation_path = os.path.join('feature_transformation_artifacts', 'geography_ohe.pkl')
scaler_path = os.path.join('feature_transformation_artifacts', 'standard_scaler.pkl')
model_path = os.path.join('model_artifacts', 'customer_churn_model.keras')

with open(gender_transformation_path, 'rb') as f:
    gender_ohe = pickle.load(f)

with open(geography_transformation_path, 'rb') as f:
    geography_ohe = pickle.load(f)

with open(scaler_path, 'rb') as f:
    scaler = pickle.load(f)

model = load_model(model_path)

In [76]:
model

<Sequential name=sequential_10, built=True>

In [77]:

# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [78]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


### Transform Geometry

In [79]:
geography_ohe.transform(input_df[['Geography']])

array([[1., 0., 0.]])

In [91]:
geography_ohe.categories_

[array(['France', 'Germany', 'Spain'], dtype=object)]

In [80]:
geography_ohe.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [81]:
geography_encoded = geography_ohe.transform(input_df[['Geography']])
pd.DataFrame(geography_encoded, columns=geography_ohe.get_feature_names_out())

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [82]:
geography_df = pd.DataFrame(geography_encoded, columns=geography_ohe.get_feature_names_out())
input_df = pd.concat([input_df, geography_df], axis=1)
input_df.drop('Geography', axis=1, inplace=True)

### Transform Gender

In [83]:
input_df[['Gender']]

,Gender
0,Male


In [84]:
gender_ohe.get_feature_names_out()

array(['Gender_Male'], dtype=object)

In [85]:
input_df[gender_ohe.get_feature_names_out()] = gender_ohe.transform(input_df[['Gender']])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Gender_Male
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0,1.0


In [86]:
input_df.drop('Gender', axis=1, inplace=True)

In [87]:
input_df

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Gender_Male
0,600,40,3,60000,2,1,1,50000,1.0,0.0,0.0,1.0


### Feature Scaling

In [88]:
scaled_data = scaler.transform(input_df)
scaled_data

array([[-0.53598516,  0.10479359, -0.69539349, -0.25781119,  0.80843615,
         0.64920267,  0.97481699, -0.87683221,  1.00150113, -0.57946723,
        -0.57638802,  0.91324755]])

### Predicting the Test set 

In [89]:
model.predict(scaled_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


array([[0.04529855]], dtype=float32)

In [90]:
h5_model_path = os.path.join('model_artifacts', 'customer_churn_model.h5')
h5_model = load_model(h5_model_path)
h5_model.predict(scaled_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


array([[0.04529855]], dtype=float32)